# IOHMM 全流程示例（特征选择 + 建模 + 计算）

本 notebook 参考 `IOHMM/experiments/spy_vol_regime.py` 与 `IOHMM/regimes`，在 `IOHMM_add` 中给出完整实现：

1. 数据准备与目标构建
2. 特征选择（相关性过滤 + IC 排序）
3. ICL 选择最优状态数 `K`
4. 滚动窗口训练与一步预测
5. 评估指标与结果保存

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

# 兼容从 IOHMM_add 目录直接启动 notebook 的场景
# 将项目根目录（cu_saf）加入 sys.path，保证绝对导入可用。
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "IOHMM_add":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name != "cu_saf":
    # 向上查找仓库根目录
    for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if p.name == "cu_saf":
            PROJECT_ROOT = p
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_preprocessing.data_adapter import YFinanceAdapter
from experiments._shared import get_canonical_rv_gk
from IOHMM.regimes.features import build_vol_iohmm_dataset
from IOHMM.regimes.iohmm import GaussianIOHMM

from sklearn.metrics import mean_squared_error, mean_absolute_error

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = E:\CodeSpace\2026_mafn\cu_saf


In [2]:
# 实验配置（与主实验口径保持一致）
TARGET_TICKER = "SPY"
EXTERNAL_TICKERS = ("TLT", "HYG", "UUP", "GLD")
START_DATE = "2018-06-01"
END_DATE = "2024-12-31"

TRAIN_WINDOW = 252
REFIT_FREQ = 21
INIT_TRAIN = 504
K_VALUES = (2, 3, 4)
SEED = 42

# 特征选择参数
CORR_THRESHOLD = 0.95
TOP_N_FEATURES = 8

OUT_DIR = str(PROJECT_ROOT / "IOHMM_add" / "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(SEED)

In [3]:
def qlike(y_true_rv: np.ndarray, y_pred_rv: np.ndarray) -> float:
    ratio = y_true_rv / np.maximum(y_pred_rv, 1e-12)
    return float(np.mean(ratio - np.log(ratio) - 1.0))


def drop_high_corr_features(df_x: pd.DataFrame, threshold: float = 0.95) -> list[str]:
    """按相关性阈值做去冗余，保留先出现的特征。"""
    corr = df_x.corr().abs()
    cols = list(corr.columns)
    keep = []
    for c in cols:
        if not keep:
            keep.append(c)
            continue
        max_corr = corr.loc[c, keep].max()
        if pd.isna(max_corr) or max_corr < threshold:
            keep.append(c)
    return keep


def select_features_by_ic(X: pd.DataFrame, y: pd.Series, top_n: int = 8) -> tuple[list[str], pd.Series]:
    """以 Spearman IC 绝对值排序选择特征。"""
    joined = X.copy()
    joined["_y_"] = y
    ic = joined.corr(method="spearman")["_y_"].drop("_y_").abs().sort_values(ascending=False)
    selected = list(ic.head(min(top_n, len(ic))).index)
    return selected, ic


def vol_pct_metrics(y_true_rv: np.ndarray, y_pred_rv: np.ndarray) -> dict:
    yt = np.sqrt(y_true_rv) * 100.0
    yp = np.sqrt(y_pred_rv) * 100.0
    return {
        "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        "mae": float(mean_absolute_error(yt, yp)),
        "qlike": qlike(y_true_rv, y_pred_rv),
    }

In [4]:
# 1) 数据准备
adapter = YFinanceAdapter()
raw = adapter.get_data(
    tickers=[TARGET_TICKER, *EXTERNAL_TICKERS],
    start_date=START_DATE,
    end_date=END_DATE,
    force_refresh=False,
)

rv_gk = get_canonical_rv_gk(ticker=TARGET_TICKER, start="2019-01-01", end=END_DATE)
log_rv_gk = np.log(rv_gk).rename("log_rv_gk")

prepared = build_vol_iohmm_dataset(
    raw,
    target_ticker=TARGET_TICKER,
    external_tickers=EXTERNAL_TICKERS,
    rv_window_external=5,
    strictly_external_inputs=True,
    target=log_rv_gk,
)

X_full = pd.DataFrame(prepared.X, index=prepared.dates, columns=prepared.feature_names)
y_full = pd.Series(prepared.y, index=prepared.dates, name="y_log_rv")

print("样本区间:", prepared.dates[0].date(), "->", prepared.dates[-1].date())
print("原始特征数:", X_full.shape[1])
X_full.head()

样本区间: 2019-01-02 -> 2024-12-31
原始特征数: 12


C:\Users\25188\AppData\Local\Temp\ipykernel_33988\3814267104.py:13: UserWarning: build_vol_iohmm_dataset: dropped 147 rows containing NaN.
  prepared = build_vol_iohmm_dataset(


,lag1_ret_TLT,lag1_log_rv_TLT,lag5_cumret_TLT,lag1_ret_HYG,lag1_log_rv_HYG,lag5_cumret_HYG,lag1_ret_UUP,lag1_log_rv_UUP,lag5_cumret_UUP,lag1_ret_GLD,lag1_log_rv_GLD,lag5_cumret_GLD
Date,,,,,,,,,,,,
2019-01-02,0.003793,-2.146804,0.006523,0.000740,-1.960390,0.010786,-0.001963,-2.777880,-0.008373,0.001568,-2.463705,0.021087
2019-01-03,0.005253,-2.141689,0.006818,-0.001358,-2.125692,0.016935,0.007438,-2.493270,0.003516,0.000660,-2.771051,0.010856
2019-01-04,0.011315,-2.690034,0.028907,0.000494,-4.021278,0.000741,-0.005475,-2.508273,-0.007034,0.009025,-2.846793,0.022885
2019-01-07,-0.011643,-1.956238,0.017098,0.016645,-2.146111,0.018003,-0.001570,-2.557088,-0.004311,-0.008119,-2.310039,0.007190
2019-01-08,-0.002953,-1.971720,0.005766,0.009663,-2.104415,0.026184,-0.004330,-2.516889,-0.005900,0.003453,-2.319292,0.006587


In [5]:
# 2) 特征选择：先去高相关，再做 IC 排序（仅在初始化训练窗口内做一次）
init_end = min(INIT_TRAIN, len(X_full))
X_init = X_full.iloc[:init_end].copy()
y_init = y_full.iloc[:init_end].copy()

corr_kept = drop_high_corr_features(X_init, threshold=CORR_THRESHOLD)
X_corr = X_init[corr_kept]

selected_features, ic_rank = select_features_by_ic(X_corr, y_init, top_n=TOP_N_FEATURES)
X_sel = X_full[selected_features].copy()

print("相关性过滤后特征数:", len(corr_kept))
print("最终选择特征数:", len(selected_features))
print("选择结果:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}  |IC|={ic_rank.loc[f]:.4f}")

feature_report = pd.DataFrame({
    "feature": ic_rank.index,
    "abs_spearman_ic": ic_rank.values,
    "selected": ic_rank.index.isin(selected_features),
})
feature_report.to_csv(f"{OUT_DIR}/feature_selection_report.csv", index=False)

相关性过滤后特征数: 12
最终选择特征数: 8
选择结果:
01. lag1_log_rv_HYG  |IC|=0.5905
02. lag1_log_rv_UUP  |IC|=0.3560
03. lag1_log_rv_GLD  |IC|=0.3031
04. lag5_cumret_TLT  |IC|=0.2470
05. lag1_log_rv_TLT  |IC|=0.2251
06. lag5_cumret_HYG  |IC|=0.1707
07. lag1_ret_TLT  |IC|=0.1078
08. lag5_cumret_GLD  |IC|=0.1064


In [6]:
# 3) 在初始化窗口做 K 选择（按 ICL）
X0 = X_sel.iloc[:init_end].to_numpy(dtype=float)
y0 = y_full.iloc[:init_end].to_numpy(dtype=float)

k_records = []
for K in K_VALUES:
    model = GaussianIOHMM(n_states=K, max_iter=100, n_init=10, random_state=SEED)
    try:
        model.fit(X0, y0)
        k_records.append({"K": K, "loglik": model.best_loglik_, "bic": model.bic_, "icl": model.icl_})
        print(f"K={K}: ll={model.best_loglik_:.2f}, BIC={model.bic_:.2f}, ICL={model.icl_:.2f}")
    except Exception as e:
        warnings.warn(f"K={K} 拟合失败: {e}")
        k_records.append({"K": K, "loglik": -np.inf, "bic": np.inf, "icl": np.inf})

k_df = pd.DataFrame(k_records).sort_values("K")
best_K = int(k_df.loc[k_df["icl"].idxmin(), "K"])
print("\n最优状态数 best_K =", best_K)
k_df.to_csv(f"{OUT_DIR}/k_selection.csv", index=False)
k_df

E:\CodeSpace\2026_mafn\cu_saf\IOHMM\regimes\iohmm.py:183: UserWarning: Transition optimization failed for prev_state=0: ABNORMAL: . Keeping previous weights.
  warnings.warn(
E:\CodeSpace\2026_mafn\cu_saf\IOHMM\regimes\iohmm.py:183: UserWarning: Transition optimization failed for prev_state=1: ABNORMAL: . Keeping previous weights.
  warnings.warn(


K=2: ll=-687.46, BIC=1605.16, ICL=2169.96


E:\CodeSpace\2026_mafn\cu_saf\IOHMM\regimes\iohmm.py:183: UserWarning: Transition optimization failed for prev_state=2: ABNORMAL: . Keeping previous weights.
  warnings.warn(


K=3: ll=-677.02, BIC=1870.51, ICL=2644.59


E:\CodeSpace\2026_mafn\cu_saf\IOHMM\regimes\iohmm.py:183: UserWarning: Transition optimization failed for prev_state=3: ABNORMAL: . Keeping previous weights.
  warnings.warn(


K=4: ll=-669.50, BIC=2253.71, ICL=3156.55

最优状态数 best_K = 2


,K,loglik,bic,icl
0,2,-687.460279,1605.155880,2169.955524
1,3,-677.018760,1870.511349,2644.590400
2,4,-669.496359,2253.711430,3156.548544


In [ ]:
# 4) 滚动训练与一步预测
X_np = X_sel.to_numpy(dtype=float)
y_np = y_full.to_numpy(dtype=float)
dates = y_full.index

pred_dates = []
y_true_log = []
y_pred_log = []
pred_var = []
dominant_state = []
state_probs = []

iohmm = GaussianIOHMM(n_states=best_K, max_iter=100, n_init=10, random_state=SEED)
refit_count = 0

for t in range(TRAIN_WINDOW, len(y_np), REFIT_FREQ):
    X_tr = X_np[t - TRAIN_WINDOW : t]
    y_tr = y_np[t - TRAIN_WINDOW : t]
    end = min(t + REFIT_FREQ, len(y_np))

    try:
        iohmm.fit(X_tr, y_tr)
        refit_count += 1
    except Exception as e:
        warnings.warn(f"t={t} 处 refit 失败: {e}")
        continue

    for i in range(t, end):
        hist_lo = i - TRAIN_WINDOW
        X_hist = X_np[hist_lo:i]
        y_hist = y_np[hist_lo:i]
        x_next = X_np[i]

        try:
            yhat, varhat, p_next = iohmm.forecast(X_hist, y_hist, x_next)
        except Exception as e:
            warnings.warn(f"t={i} 处预测失败: {e}")
            continue

        pred_dates.append(dates[i])
        y_true_log.append(y_np[i])
        y_pred_log.append(yhat)
        pred_var.append(varhat)
        dominant_state.append(int(np.argmax(p_next)))
        state_probs.append(p_next)

print("完成 refit 次数:", refit_count)
print("生成预测条数:", len(y_pred_log))

E:\CodeSpace\2026_mafn\cu_saf\IOHMM\regimes\iohmm.py:183: UserWarning: Transition optimization failed for prev_state=0: ABNORMAL: . Keeping previous weights.
  warnings.warn(
E:\CodeSpace\2026_mafn\cu_saf\IOHMM\regimes\iohmm.py:183: UserWarning: Transition optimization failed for prev_state=1: ABNORMAL: . Keeping previous weights.
  warnings.warn(


In [ ]:
# 5) 指标计算与结果保存
pred_df = pd.DataFrame(
    {
        "date": pd.DatetimeIndex(pred_dates),
        "y_true_log": np.asarray(y_true_log, dtype=float),
        "y_pred_log": np.asarray(y_pred_log, dtype=float),
        "pred_var": np.asarray(pred_var, dtype=float),
        "dom_state": np.asarray(dominant_state, dtype=int),
    }
).sort_values("date").reset_index(drop=True)

for k in range(best_K):
    pred_df[f"p_state_{k}"] = [p[k] for p in state_probs]

pred_df["y_true_rv"] = np.exp(pred_df["y_true_log"])
pred_df["y_pred_rv"] = np.exp(pred_df["y_pred_log"])

valid = np.isfinite(pred_df["y_true_rv"].to_numpy()) & np.isfinite(pred_df["y_pred_rv"].to_numpy())
eval_df = pred_df.loc[valid].copy()

metrics = vol_pct_metrics(eval_df["y_true_rv"].to_numpy(), eval_df["y_pred_rv"].to_numpy())
metrics["n"] = int(len(eval_df))
metrics["best_K"] = best_K
metrics["n_features"] = len(selected_features)

metrics_df = pd.DataFrame([metrics])

pred_df.to_csv(f"{OUT_DIR}/iohmm_predictions.csv", index=False)
metrics_df.to_csv(f"{OUT_DIR}/iohmm_metrics.csv", index=False)

print("评估结果:")
print(metrics_df.to_string(index=False))
print("\n文件已保存到:", OUT_DIR)
pred_df.head()

In [ ]:
# 可选：按状态查看误差
regime_rows = []
for k in range(best_K):
    tmp = eval_df[eval_df["dom_state"] == k]
    if len(tmp) < 5:
        regime_rows.append({"state": k, "n": len(tmp), "rmse": np.nan, "mae": np.nan, "qlike": np.nan})
        continue
    mm = vol_pct_metrics(tmp["y_true_rv"].to_numpy(), tmp["y_pred_rv"].to_numpy())
    regime_rows.append({"state": k, "n": len(tmp), **mm})

regime_df = pd.DataFrame(regime_rows)
regime_df.to_csv(f"{OUT_DIR}/iohmm_regime_metrics.csv", index=False)
regime_df